# Python Tutorial for Beginner
初心者を対象に、pythonを用いて、機械学習の一連の流れを理解することを目的とする。

### モジュールのインストール
コマンドプロンプトではなく、 jupyter notebook で起動する場合は、  ! をつかう。

In [ ]:
!pip install pandas
!pip install numpy
!pip install matplotlib

#Google Colab用
! wget https://github.com/ChemicalBatteryLab-Nitech/dxgem_day4day5/blob/main/Day4_Part1/olivineDataset_withEA.csv

### モジュールのインポート
インストールしたモジュールをロードする

In [ ]:
import pandas

In [ ]:
pandas


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

In [ ]:
pd

In [ ]:
df=pd.DataFrame([[1,2],[3,4]])
display(df)
type(df)

### Pandas Dataframeの使い方

In [ ]:
# CSVファイルからの読み込み
df = pd.read_csv('./olivineDataset_withEA.csv')
df

In [ ]:
#空白があるデータを削除する（指定がなければ、行の情報が消える）
df1=df.dropna()
#df1a=df.dropna(axis=1)
df1

In [ ]:
df2=df.iloc[0:3,2:5]
print(df2)
df3=df.loc[0:5,'a':'c']
print(df3)

In [ ]:
#  記述子(descriptor)と目的変数(objective)の抽出

#df1を使う（目的変数空白の行は訓練データにならない）
#元のデータフレームから、必要ない列を削除する。
#データフレームから、列を削除する時は、df.drop（[削除する列のリスト], axis=0 or 1 行を消す時は0、列の時は1）

drop_columns = ['Composition', 'EA (eV)', 'S/U']
df_descriptor_all = df.drop(drop_columns, axis=1)
df_descriptor = df1.drop(drop_columns, axis=1)
display(df_descriptor)
df_objective = df1['EA (eV)']
display(df_objective)

### 先走って機械学習

In [ ]:
# PLS regressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_squared_error, r2_score

# PLSモデル（成分数を指定：通常は 2〜10くらいから検討）
pls = PLSRegression(n_components=4)
pls.fit(df_descriptor, df_objective)  # Pandas DataFrameを直接使用できる
# 予測
y_pred_temp = pls.predict(df_descriptor)

# 診断プロット
plt.scatter(df_objective, y_pred_temp, color='blue', label='PLS Prediction')
plt.xlabel('True EA (eV)')
plt.ylabel('Predicted EA (eV)')
plt.title('PLS Regression Prediction')
plt.plot([df_objective.min(), df_objective.max()], [df_objective.min(), df_objective.max()], color='red', linestyle='--', label='y=x')

# 評価
print("MSE:", mean_squared_error(df_objective, y_pred_temp), "eV^2")
print("RMSE:", (mean_squared_error(df_objective, y_pred_temp))**0.5, "eV")
print("R² Score:", r2_score(df_objective, y_pred_temp))

### 【参考】よく使う Pandas処理

In [ ]:
df1 = df_descriptor.copy()
print('=====処理前=====')
display(df1.head())

#新しい記述子を追加したいとき
#今回は例として、a,b,c軸の格子定数の和の記述子を追加してみる。
df1['a+b+c'] = df1['a'] + df1['b'] + df1['c']
print('=====a+b+cの列を追加=====')
display(df1.head())

#データフレームをスライスする。
#列の名前で指定
df2 = df1.loc[:, 'a':'c']
print('=====a~c列までをスライス（列名で指定ver）=====')
display(df2.head())
#列の列番号で指定
df3 = df1.iloc[:, 0:3]
print('=====a~c列までをスライス（列番号で指定ver）=====')
display(df3.head())

## Numpy 行列計算モジュール

In [ ]:
a=np.array([[0,1],[2,3]])
a

In [ ]:
print(np.mean(a))
print(np.std(a))
print(np.sum(a))

In [ ]:
print(a*a)
print(np.dot(a,a))

from numpy.linalg import inv
print("逆行列---")
print(inv(a))        # 逆行列

print("乱数生成---")
np.random.seed(0)           # 乱数の再現性
print(np.random.rand(3))    # 一様乱数 [0,1)
print(np.random.randn(3))   # 正規分布

### Pandas Numpy 相互変換
Numpyでは行・列ラベルが失われることに注意

In [ ]:
arr_descriptor=df_descriptor.to_numpy()
print(arr_descriptor)

df_r=pd.DataFrame(arr_descriptor,index=df_descriptor.index, columns=df_descriptor.columns)
df_r

### Scikit Learnによる機械学習回帰分析 
1. 訓練・テストデータ分離<BR>

In [ ]:
!pip install scikit-learn

In [ ]:
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
train_test_split

In [ ]:
#test_size : テストデータの大きさ
#random_state : 分割の再現性を保つため
X_train, X_test, y_train, y_test = train_test_split(df_descriptor, df_objective, test_size=0.3, random_state=42) #テストデータとトレーニングデータの分割

print('=====訓練用データ=====')
display(X_train)
display(y_train)
print('=====検証用データ=====')
display(X_test)
display(y_test)

2. 記述子尺度調整 (標準化またはmaxmin処理)<BR>


In [ ]:
#x = (x-mean)/std
#データ標準化用のモジュール
from sklearn.preprocessing import StandardScaler

#インスタンス化
scaler = StandardScaler()

#scaler.fit(処理前のデータ)で、各列の平均値mean、標準偏差stdを学習する。
# 注　後でこの scalerに記録された平均値、標準偏差をdropnaで切り落としたデータにも使うので、消去しないこと。
scaler.fit(X_train)

#scaler.transform(処理前のデータ)で、標準化処理されたデータがnumpy形式で返される。
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('=====BEFORE=====')
display(X_train.head())
print('=====AFTER=====')
print(X_train_scaled[0:2,:])
#numpyで返されているので、可視化用にデータフレームに変換
df_X_train_scaled = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
display(df_X_train_scaled.head())

3. 機械学習回帰分析(PLS, RandomForest)<BR>

In [ ]:
# PLS regressor

from sklearn.cross_decomposition import PLSRegression
# PLSモデル（成分数を指定：通常は 2〜10くらいから検討）
pls = PLSRegression(n_components=4)
pls.fit(X_train_scaled, y_train)
# 予測
y_pred_pls = pls.predict(X_test_scaled)

# 評価
print("MSE:", mean_squared_error(y_test, y_pred_pls), "eV^2")
print("RMSE:", (mean_squared_error(y_test, y_pred_pls))**0.5, "eV")
print("R² Score:", r2_score(y_test, y_pred_pls))

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# --- ランダムフォレスト回帰モデルの学習 ---
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# --- テストデータで予測 ---
y_pred_RF = model.predict(X_test_scaled)

# 評価
print("MSE:", mean_squared_error(y_test, y_pred_RF), "eV^2")
print("RMSE:", (mean_squared_error(y_test, y_pred_RF))**0.5, "eV")
print("R² Score:", r2_score(y_test, y_pred_RF))

<Question>

### 現実的な機械学習回帰分析のために
1. ｋ分割交差検証<BR>
2. ハイパラチューニング<BR>


【Qustion to ChatGPT】
```python
        # PLS regressor
        from sklearn.cross_decomposition import PLSRegression
        # PLSモデル（成分数を指定：通常は 2〜10くらいから検討）
        pls = PLSRegression(n_components=4)
        pls.fit(X_train_scaled, y_train)
        # 予測
        y_pred_pls = pls.predict(X_test_scaled)
        # 評価
        print("MSE:", mean_squared_error(y_test, y_pred_pls), "eV^2")
        print("RMSE:", (mean_squared_error(y_test, y_pred_pls))**0.5, "eV")
        print("R² Score:", r2_score(y_test, y_pred_pls))
```


のスクリプトをベースに、ｋ分割交差検証（Ｋ＝５）を行い、主成分数を2~10に変化させて最適値を調整してください。その後、テストデータでR2などを評価するようにしてください。<BR>

【Answer from ChatGPT】<BR>
以下は、ご要望に沿って：<BR>
K=5の交差検証（cross-validation）<BR>
主成分数を2〜10で変化させて、最も良いn_componentsを選択<BR>
最終的にそのn_componentsでモデルを再学習し、テストデータで評価<BR>
を行うスクリプトです。前提として、X_train_scaled, X_test_scaled, y_train, y_test がすでに定義されているとします。<BR>


In [ ]:
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# 交差検証：主成分数 2~10 を試す
best_score = -np.inf
best_n = None
cv_scores = []

for n in range(2, 11):
    pls = PLSRegression(n_components=n)
    # R²スコア（cv=5）で交差検証（スコア平均）
    scores = cross_val_score(pls, X_train_scaled, y_train, cv=5, scoring='r2')
    mean_score = np.mean(scores)
    cv_scores.append(mean_score)

    print(f"n_components={n}, CV R²={mean_score:.4f}")
    if mean_score > best_score:
        best_score = mean_score
        best_n = n

print(f"\n✅ 最も良かった主成分数: {best_n} (CV R²={best_score:.4f})")

# 最良の成分数で学習し直し → テストデータで予測
pls_best = PLSRegression(n_components=best_n)
pls_best.fit(X_train_scaled, y_train)
y_pred_pls = pls_best.predict(X_test_scaled)

# テストデータでの評価
mse = mean_squared_error(y_test, y_pred_pls)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_pls)

print("\n📊 テストデータ評価:")
print(f"MSE:  {mse:.4f} eV^2")
print(f"RMSE: {rmse:.4f} eV")
print(f"R²:   {r2:.4f}")

### fitting 結果の保存と読み込み
今回の場合、機械学習の予測関数のみではなく、データの標準化についても保管する必要がある。（再現性のため）

In [ ]:
from sklearn.pipeline import make_pipeline
import joblib

# パイプライン作成（scaler + PLS）
pipeline = make_pipeline(scaler, pls_best)

# パイプラインごと保存
joblib.dump(pipeline, 'pls_pipeline_with_scaler.joblib')
print("✅ モデル＋スケーラーを保存しました: pls_pipeline_with_scaler.joblib")



In [ ]:
# モデルとスケーラーの読み込み
loaded_pipeline = joblib.load('pls_pipeline_with_scaler.joblib')

# 新しい入力データ（未スケーリング）で予測可能
y_pred_new = loaded_pipeline.predict(X_test)  # X_test は未標準化のままでOK


### Fitting可視化 (Matplotlib)

In [ ]:
import matplotlib.pyplot as plt

y_pred_train = pls_best.predict(X_train_scaled)

# Parity plot: Actual vs Predicted (Test)
plt.figure()
plt.scatter(y_train, y_pred_train)
plt.scatter(y_test, y_pred_pls)
min_val = min(y_test.min(), y_pred_pls.min())
max_val = max(y_test.max(), y_pred_pls.max())
plt.plot([0, 1.5], [0, 1.5])  # y=x line
plt.xlabel("Actual EA (eV)")
plt.ylabel("Predicted EA (eV)")
plt.title("Parity Plot (Test Set)")
plt.tight_layout()


### 全データについて予測実施

In [ ]:
y_all_pred=pls_best.predict(scaler.transform(df_descriptor_all.to_numpy()))
y_all_pred

In [ ]:
df['Predicted_EA'] = y_all_pred
df